# Declare Co-produced Outputs

Emit a dataset and its metric in one creator call. The metric points to the exact output position returned by `add_artifact`, even when two outputs have equal contents. These are direct associations; a matching filename alone creates no relation.

In [ ]:
from __future__ import annotations

import csv
import io
import json
import random
import statistics
from enum import StrEnum
from typing import Any, ClassVar

from pydantic import BaseModel, Field

from artisan.execution import Executor
from artisan.operations.base import OperationDefinition
from artisan.schemas import (
    ArtifactResult,
    CSVArtifact,
    ExecuteInput,
    MetricArtifact,
    OutputSpec,
    PostprocessInput,
)

In [ ]:
class DatasetSummarizer(OperationDefinition):
    name = "tutorial_dataset_summarizer"
    description = "Generate datasets and directly associated summary metrics"

    class OutputRole(StrEnum):
        datasets = "datasets"
        metrics = "metrics"

    outputs: ClassVar[dict[str, OutputSpec]] = {
        "datasets": OutputSpec(artifact_type="csv", derives_from={"inputs": []}),
        "metrics": OutputSpec(
            artifact_type="metric", derives_from={"outputs": ["datasets"]}
        ),
    }

    class Params(BaseModel):
        count: int = Field(default=3, ge=1, description="Number of datasets")
        rows: int = Field(default=8, ge=1, description="Rows per dataset")
        seed: int = Field(default=42, description="Deterministic random seed")

    params: Params = Params()

    def execute_function(self, _inputs: ExecuteInput) -> dict[str, Any]:
        random_source = random.Random(self.params.seed)
        outputs = []
        for index in range(self.params.count):
            scores = [round(random_source.random(), 4) for _ in range(self.params.rows)]
            stream = io.StringIO()
            writer = csv.writer(stream)
            writer.writerow(["id", "score"])
            writer.writerows(enumerate(scores))
            outputs.append(
                {
                    "content": stream.getvalue().encode(),
                    "mean": statistics.mean(scores),
                    "name": f"dataset_{index:05d}",
                }
            )
        return {"outputs": outputs}

    def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
        result = ArtifactResult()
        for item in inputs.memory_outputs["outputs"]:
            position = result.add_artifact(
                "datasets",
                CSVArtifact.draft(
                    content=item["content"], original_name=item["name"] + ".csv"
                ),
                sources={},
            )
            result.add_artifact(
                "metrics",
                MetricArtifact.draft(
                    content={"mean_score": item["mean"], "row_count": self.params.rows},
                    original_name=item["name"] + "_stats",
                ),
                sources={"datasets": [position]},
            )
        return result

In [ ]:
with Executor() as executor:
    result = executor.run(DatasetSummarizer)
    datasets = result.artifacts["datasets"]
    metrics = result.artifacts["metrics"]
    assert len(datasets) == len(metrics) == 3
    assert [edge.source_output_index for edge in result.lineage["metrics"]] == [0, 1, 2]
    for dataset, metric in zip(datasets, metrics, strict=True):
        rows = list(csv.DictReader(io.StringIO(dataset.content.decode())))
        values = json.loads(metric.content)
        assert values["row_count"] == len(rows) == 8
        assert values["mean_score"] == statistics.mean(
            float(row["score"]) for row in rows
        )
assert executor.cleanup_complete
print("Every metric matches the bytes of its declared dataset parent")

A durable pipeline uses the same output declarations after this operation is moved into an installed application package. The ordinary provenance view then exposes these direct edges; it does not infer them from content IDs or names.